In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import expr
from pyspark.sql.types import StringType
from delta.tables import DeltaTable

In [0]:
# Paths
bronze_path = '/Volumes/scd2/volume/bronze'
silver_path = '/Volumes/scd2/volume/silver/scd2_data'
checkpoint_path = '/Volumes/scd2/volume/silver/_checkpoints'

In [0]:
# Read bronze data as stream (automatically incremental)
bronze_stream = spark.readStream.format('delta').load(bronze_path)

# Select only clean columns
clean_cols = ['pk', 'name', 'email', 'city', 'status', 'event_time']
bronze_clean = bronze_stream.select(*clean_cols)

In [0]:

def process_batch(batch_df, batch_id):
    # Deduplicate: keep latest record per pk
   
    REQUIRED_NON_NULL_COLS = ["pk", "event_time", "name", "email", "city", "status"]
    batch_df=batch_df.select(*REQUIRED_NON_NULL_COLS)
    valid_df = batch_df.dropna(subset=REQUIRED_NON_NULL_COLS, how="any")
    batch_df = (valid_df
        .withColumn(
            "event_time",
            expr("coalesce(try_to_timestamp(event_time, 'yyyy-MM-dd HH:mm:ss'), "
                 "try_to_timestamp(event_time, 'M/d/yyyy H:mm'))")
        )
        .filter(F.col("event_time").isNotNull())
    )
    
    window_spec = Window.partitionBy('pk').orderBy(F.col('event_time').desc())
    
    filtered_df = (batch_df
        .withColumn('row_num', F.row_number().over(window_spec))
        .filter(F.col('row_num') == 1)
        .drop('row_num')
    )
    
    # Add SCD2 columns
    updates_df = (filtered_df
        .withColumn('effective_date', F.to_timestamp('event_time'))
        .withColumn('end_date', F.lit(None).cast('timestamp'))
        .withColumn('is_current', F.lit(True))
        .withColumn(
            "record_hash",
            F.sha2(F.concat_ws("||",
                F.coalesce(F.col("name"), F.lit("")),
                F.coalesce(F.col("email"), F.lit("")),
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("status"), F.lit(""))
            ), 256)
        )
        .select('pk', 'name', 'email', 'city', 'status', 'event_time', 'effective_date', 'end_date', 'is_current','record_hash')
    )
    
    # Check if silver table exists
    if DeltaTable.isDeltaTable(spark, silver_path):
        silver_table = DeltaTable.forPath(spark, silver_path)
        
        isactive = (silver_table.toDF()
            .filter("is_current = True")
            .select(F.col('pk').alias('s_pk'), F.col('record_hash').alias('s_hash')))

        changed = (updates_df.alias('u')
            .join(isactive.alias('s'), F.col("u.pk") == F.col("s.s_pk"), "inner")
            .filter(F.col('s.s_hash') != F.col('u.record_hash'))
            .select('u.*'))

        staged = (updates_df.withColumn("Mergekey", F.col('pk'))
            .unionByName(changed.withColumn('Mergekey', F.lit(None).cast(StringType()))))

        (silver_table.alias('s')
            .merge(staged.alias('u'), "s.pk = u.Mergekey")
            .whenMatchedUpdate(
                condition="""s.record_hash <> u.record_hash 
                    and u.effective_date > s.effective_date 
                    and s.is_current = True""",
                set={"is_current": "false", "end_date": "u.effective_date"}
            )
            .whenNotMatchedInsert(
                values={
                    "pk": "u.pk", "name": "u.name", "email": "u.email",
                    "city": "u.city", "status": "u.status", "event_time": "u.event_time",
                    "effective_date": "u.effective_date",
                    "end_date": "cast(null as timestamp)",
                    "is_current": "True", "record_hash": "u.record_hash"
                }
            )
            .execute())
    else:
        # Initial load only — table genuinely doesn't exist yet
        updates_df.write.format('delta').mode('overwrite').save(silver_path)

In [0]:
# Start streaming query with foreachBatch
query = (bronze_clean.writeStream
    .foreachBatch(process_batch)
    .option('checkpointLocation', checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

In [0]:
# Read and display silver table
silver_final = spark.read.format('delta').load(silver_path)


In [0]:
silver_final.count()

In [0]:
silver_final.display()